# Import

In [29]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import StandardScaler, OrdinalEncoder

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

from industrial_predictive_maintenance.data import load_data

# Load data

In [17]:
df = load_data("../data/raw/ai4i2020Raw.csv")

In [18]:
df.head()

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0


# Preprocess

In [19]:
df = df.drop(columns=["UDI",
     "Product ID", "TWF",
     "HDF",
     "PWF",
     "OSF",
     "RNF",])

In [20]:
df.columns = df.columns.str.lower().str.replace(r'\s*\[.*?\]$', '', regex=True).str.replace(" ", "_")

In [21]:
df.head()

,type,air_temperature,process_temperature,rotational_speed,torque,tool_wear,machine_failure
0,M,298.1,308.6,1551,42.8,0,0
1,L,298.2,308.7,1408,46.3,3,0
2,L,298.1,308.5,1498,49.4,5,0
3,L,298.2,308.6,1433,39.5,7,0
4,L,298.2,308.7,1408,40.0,9,0


In [22]:
df["temperature_difference"] = df["process_temperature"] - df["air_temperature"]

In [23]:
df["power"] = df["torque"] * df["rotational_speed"]

In [24]:
df.head()

,type,air_temperature,process_temperature,rotational_speed,torque,tool_wear,machine_failure,temperature_difference,power
0,M,298.1,308.6,1551,42.8,0,0,10.5,66382.8
1,L,298.2,308.7,1408,46.3,3,0,10.5,65190.4
2,L,298.1,308.5,1498,49.4,5,0,10.4,74001.2
3,L,298.2,308.6,1433,39.5,7,0,10.4,56603.5
4,L,298.2,308.7,1408,40.0,9,0,10.5,56320.0


# Data split

In [27]:
X = df.drop("machine_failure", axis=1)
y = df["machine_failure"]

In [28]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.15, random_state=101, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1765, random_state=101, stratify=y_train) # ≈15% of the original data 

# Transform

In [30]:
# Encode
encoder = OrdinalEncoder(categories=[["L", "M", "H"]])

X_train["type"] = encoder.fit_transform(X_train[["type"]])
X_test["type"] = encoder.transform(X_test[["type"]])
X_val["type"] = encoder.transform(X_val[["type"]])

# Scale numeric columns
numeric_cols = ["air_temperature", "process_temperature", "rotational_speed", "torque", "tool_wear", "temperature_difference", "power"]

scaler = StandardScaler()

X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])
X_val[numeric_cols] = scaler.transform(X_val[numeric_cols])

In [31]:
X_train.head()

,type,air_temperature,process_temperature,rotational_speed,torque,tool_wear,temperature_difference,power
2364,0.0,-0.409716,-1.022638,-0.960786,0.970052,-0.538354,-0.698769,0.776256
4473,0.0,1.339615,0.324962,0.073361,-0.412592,-1.609933,-2.200483,-0.422044
9491,1.0,-0.509678,-0.214078,-0.273218,0.068328,1.636321,0.702831,0.060974
2646,1.0,-0.109831,-0.348838,-0.474457,0.318807,0.706569,-0.298312,0.273687
5942,0.0,0.339997,0.459722,-0.491227,0.338845,-0.443803,0.002031,0.289430


# Model

## Logistic Regression